In [ ]:
# %% [markdown]
# # Notebook 01: Data Acquisition & Initial Exploration
# ## VeritasFinancial - Banking Fraud Detection System
# 
# **Objective**: Load, understand, and perform initial quality assessment of banking transaction data
# 
# **Key Tasks**:
# 1. Connect to data sources (CSV, database, APIs)
# 2. Load transaction data with proper data types
# 3. Perform initial data quality checks
# 4. Document data dictionary
# 5. Save processed data for further analysis

# %% [markdown]
# ### 1. Import Required Libraries

# %%
# Core data science libraries
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical operations
import matplotlib.pyplot as plt  # For basic plotting
import seaborn as sns  # For statistical visualizations

# Database and file handling
from sqlalchemy import create_engine, text  # For database connections
import pyarrow.parquet as pq  # For efficient data storage
import os  # For operating system operations
from pathlib import Path  # For path manipulation

# Data validation
import pandera as pa  # For data schema validation
from pandera import Check, Column, DataFrameSchema  # Schema definition

# Utilities
import warnings  # For managing warnings
from datetime import datetime  # For timestamp handling
import logging  # For logging operations
import yaml  # For configuration files
import json  # For JSON handling

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up logging to track all operations
logging.basicConfig(
    level=logging.INFO,  # Log INFO level and above
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',  # Log format
    handlers=[
        logging.FileHandler('data_acquisition.log'),  # Save to file
        logging.StreamHandler()  # Print to console
    ]
)
logger = logging.getLogger(__name__)  # Create logger instance

# Set random seeds for reproducibility
np.random.seed(42)

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')  # Professional plotting style
sns.set_palette("husl")  # Color palette for consistency
%matplotlib inline  # Display plots inline in notebook

# %% [markdown]
# ### 2. Load Configuration

# %%
# Load configuration from YAML file
def load_config(config_path='../configs/data_config.yaml'):
    """
    Load configuration settings from YAML file.
    
    Args:
        config_path (str): Path to configuration file
        
    Returns:
        dict: Configuration dictionary
    """
    try:
        with open(config_path, 'r') as file:
            config = yaml.safe_load(file)
        logger.info(f"Configuration loaded successfully from {config_path}")
        return config
    except FileNotFoundError:
        logger.warning(f"Config file not found at {config_path}. Using default settings.")
        # Return default configuration if file doesn't exist
        return {
            'data_sources': {
                'transactions': {
                    'type': 'csv',
                    'path': '../data/raw/creditcard.csv'
                }
            },
            'data_quality': {
                'missing_value_threshold': 0.3,
                'outlier_method': 'iqr',
                'iqr_multiplier': 1.5
            }
        }

# Load configuration
config = load_config()
logger.info("Configuration loaded successfully")

# Display configuration
print("=" * 60)
print("CONFIGURATION SETTINGS")
print("=" * 60)
print(json.dumps(config, indent=2))

# %% [markdown]
# ### 3. Data Source Connection Functions

# %%
class DataConnector:
    """
    Unified data connector class for handling multiple data sources.
    Supports CSV, Parquet, Excel, JSON, PostgreSQL, and API connections.
    """
    
    def __init__(self, config):
        """
        Initialize data connector with configuration.
        
        Args:
            config (dict): Configuration dictionary
        """
        self.config = config
        self.connections = {}  # Store active connections
        logger.info("DataConnector initialized")
    
    def connect_to_database(self, connection_string, **kwargs):
        """
        Establish connection to database.
        
        Args:
            connection_string (str): Database connection string
            **kwargs: Additional connection parameters
            
        Returns:
            sqlalchemy.engine.Engine: Database engine
        """
        try:
            engine = create_engine(connection_string, **kwargs)
            # Test connection
            with engine.connect() as conn:
                conn.execute(text("SELECT 1"))
            logger.info(f"Database connection established successfully")
            return engine
        except Exception as e:
            logger.error(f"Database connection failed: {str(e)}")
            raise
    
    def load_csv(self, file_path, **kwargs):
        """
        Load data from CSV file.
        
        Args:
            file_path (str): Path to CSV file
            **kwargs: Additional pandas read_csv parameters
            
        Returns:
            pd.DataFrame: Loaded data
        """
        try:
            # Default parameters for optimal CSV reading
            default_params = {
                'low_memory': False,  # Read all columns at once for type inference
                'memory_map': True,   # Memory map for large files
                'parse_dates': True    # Parse dates automatically
            }
            # Update with user-provided parameters
            default_params.update(kwargs)
            
            df = pd.read_csv(file_path, **default_params)
            logger.info(f"CSV loaded successfully from {file_path}. Shape: {df.shape}")
            return df
        except Exception as e:
            logger.error(f"CSV loading failed: {str(e)}")
            raise
    
    def load_parquet(self, file_path, **kwargs):
        """
        Load data from Parquet file.
        
        Args:
            file_path (str): Path to Parquet file
            **kwargs: Additional pandas read_parquet parameters
            
        Returns:
            pd.DataFrame: Loaded data
        """
        try:
            df = pd.read_parquet(file_path, **kwargs)
            logger.info(f"Parquet loaded successfully from {file_path}. Shape: {df.shape}")
            return df
        except Exception as e:
            logger.error(f"Parquet loading failed: {str(e)}")
            raise
    
    def load_from_database(self, query, connection_string=None, **kwargs):
        """
        Load data from database using SQL query.
        
        Args:
            query (str): SQL query
            connection_string (str): Database connection string
            **kwargs: Additional pandas read_sql parameters
            
        Returns:
            pd.DataFrame: Loaded data
        """
        try:
            engine = self.connect_to_database(connection_string)
            df = pd.read_sql(query, engine, **kwargs)
            logger.info(f"Data loaded from database. Shape: {df.shape}")
            return df
        except Exception as e:
            logger.error(f"Database loading failed: {str(e)}")
            raise
    
    def load_from_api(self, url, params=None, headers=None, **kwargs):
        """
        Load data from API endpoint.
        
        Args:
            url (str): API endpoint URL
            params (dict): Query parameters
            headers (dict): HTTP headers
            **kwargs: Additional request parameters
            
        Returns:
            pd.DataFrame: Loaded data
        """
        try:
            import requests
            response = requests.get(url, params=params, headers=headers, **kwargs)
            response.raise_for_status()  # Raise exception for bad status codes
            
            # Try to parse JSON response
            data = response.json()
            
            # Handle different response formats
            if isinstance(data, list):
                df = pd.DataFrame(data)
            elif isinstance(data, dict):
                # Try to find the data array in response
                if 'data' in data:
                    df = pd.DataFrame(data['data'])
                elif 'results' in data:
                    df = pd.DataFrame(data['results'])
                else:
                    df = pd.DataFrame([data])
            else:
                raise ValueError(f"Unexpected API response format: {type(data)}")
            
            logger.info(f"Data loaded from API. Shape: {df.shape}")
            return df
        except Exception as e:
            logger.error(f"API loading failed: {str(e)}")
            raise

# Initialize data connector
connector = DataConnector(config)

# %% [markdown]
# ### 4. Load Transaction Data

# %%
# Define data source based on configuration
data_source = config.get('data_sources', {}).get('transactions', {})

# Load the data
if data_source.get('type') == 'csv':
    # For this example, we'll use the Kaggle Credit Card Fraud dataset
    # This dataset is commonly used for fraud detection benchmarks
    file_path = data_source.get('path', '../data/raw/creditcard.csv')
    
    # Check if file exists, if not download it
    if not os.path.exists(file_path):
        logger.info("Dataset not found locally. Downloading...")
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        
        # Download from Kaggle (requires API key)
        # For this example, we'll create synthetic data if download fails
        try:
            import opendatasets as od
            od.download("https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud")
            df = pd.read_csv("creditcardfraud/creditcard.csv")
            # Move to correct location
            os.rename("creditcardfraud/creditcard.csv", file_path)
        except:
            logger.warning("Kaggle download failed. Creating synthetic data...")
            # Create synthetic data based on dataset characteristics
            n_samples = 284807  # Size of original dataset
            n_features = 30     # Number of V features
            
            np.random.seed(42)
            # Create PCA-transformed features (V1-V28)
            data = {}
            for i in range(1, 29):
                # Generate features with different distributions
                if i in [1, 2, 3, 4, 5]:  # More discriminative features
                    data[f'V{i}'] = np.random.normal(0, 1, n_samples)
                    # Add some fraud pattern
                    fraud_idx = np.random.choice(n_samples, size=int(n_samples*0.0017), replace=False)
                    data[f'V{i}'][fraud_idx] += np.random.normal(3, 1, len(fraud_idx))
                else:
                    data[f'V{i}'] = np.random.normal(0, 1, n_samples)
            
            # Add Time and Amount
            data['Time'] = np.random.randint(0, 172800, n_samples)  # Seconds in 2 days
            data['Amount'] = np.random.exponential(50, n_samples)
            data['Amount'] = np.clip(data['Amount'], 1, 20000)
            
            # Create fraud labels (0.172% fraud rate as in original)
            data['Class'] = 0
            fraud_idx = np.random.choice(n_samples, size=int(n_samples*0.00172), replace=False)
            data['Class'][fraud_idx] = 1
            
            df = pd.DataFrame(data)
            df.to_csv(file_path, index=False)
            logger.info(f"Synthetic data created and saved to {file_path}")
    
    # Load the data
    df = connector.load_csv(file_path)
    
elif data_source.get('type') == 'database':
    # Load from database
    connection_string = data_source.get('connection_string')
    query = data_source.get('query', 'SELECT * FROM transactions')
    df = connector.load_from_database(query, connection_string)
    
elif data_source.get('type') == 'api':
    # Load from API
    url = data_source.get('url')
    df = connector.load_from_api(url)
    
else:
    # Default to creating synthetic data for demonstration
    logger.warning("No valid data source specified. Creating sample data...")
    from sklearn.datasets import make_classification
    
    # Generate synthetic transaction data
    n_samples = 100000
    n_features = 28
    
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=20,
        n_redundant=5,
        n_repeated=3,
        n_clusters_per_class=1,
        weights=[0.998, 0.002],  # Highly imbalanced (0.2% fraud)
        flip_y=0.01,
        random_state=42
    )
    
    # Create DataFrame
    feature_names = [f'V{i}' for i in range(1, n_features+1)]
    df = pd.DataFrame(X, columns=feature_names)
    
    # Add Time and Amount
    df['Time'] = np.random.randint(0, 172800, n_samples)
    df['Amount'] = np.random.exponential(50, n_samples) * (1 + 0.5 * y)  # Fraud amounts slightly higher
    df['Amount'] = np.clip(df['Amount'], 1, 20000)
    df['Class'] = y
    
    logger.info(f"Sample data created. Shape: {df.shape}")

# Display basic information
print("\n" + "="*60)
print("DATA LOADING SUMMARY")
print("="*60)
print(f"Dataset shape: {df.shape}")
print(f"Number of transactions: {df.shape[0]:,}")
print(f"Number of features: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# %% [markdown]
# ### 5. Data Schema Validation

# %%
# Define data schema using pandera
def define_schema():
    """
    Define the expected schema for transaction data.
    
    Returns:
        DataFrameSchema: Pandera schema object
    """
    schema = DataFrameSchema({
        # V1-V28 features (PCA transformed, should be numerical with no missing values)
        **{f'V{i}': Column(
            float, 
            nullable=False,
            checks=[
                Check(lambda x: pd.notna(x), element_wise=False),  # No missing values
                Check(lambda x: np.isfinite(x), element_wise=False)  # Finite values
            ],
            description=f"PCA transformed feature {i}"
        ) for i in range(1, 29)},
        
        # Time feature (seconds elapsed between transactions)
        'Time': Column(
            float,
            nullable=False,
            checks=[
                Check(lambda x: x >= 0, element_wise=False),  # Time cannot be negative
                Check(lambda x: x <= 172800, element_wise=False)  # Max 2 days in seconds
            ],
            description="Seconds elapsed between this transaction and first transaction"
        ),
        
        # Amount feature
        'Amount': Column(
            float,
            nullable=False,
            checks=[
                Check(lambda x: x >= 0, element_wise=False),  # Amount cannot be negative
                Check(lambda x: x <= 100000, element_wise=False)  # Reasonable max amount
            ],
            description="Transaction amount"
        ),
        
        # Class label (0 = non-fraud, 1 = fraud)
        'Class': Column(
            int,
            nullable=False,
            checks=[
                Check(lambda x: x.isin([0, 1]), element_wise=False)  # Binary classification
            ],
            description="Fraud label: 0=non-fraud, 1=fraud"
        )
    }, strict=True)  # Enforce all columns
    
    return schema

# Define schema
schema = define_schema()

# Validate data against schema
try:
    validated_df = schema.validate(df, lazy=False)
    logger.info("Data schema validation passed successfully!")
    print("\n✅ Data schema validation passed!")
except pa.errors.SchemaError as e:
    logger.error(f"Schema validation failed: {str(e)}")
    print(f"\n❌ Schema validation failed: {str(e)}")
    # Print detailed validation report
    validation_results = schema.validate(df, lazy=True)
    print("\nValidation errors:")
    for error in validation_results.errors:
        print(f"  - {error}")

# %% [markdown]
# ### 6. Initial Data Quality Assessment

# %%
def assess_data_quality(df):
    """
    Comprehensive data quality assessment.
    
    Args:
        df (pd.DataFrame): Input dataframe
        
    Returns:
        dict: Quality metrics
    """
    quality_report = {}
    
    # 1. Missing values analysis
    print("\n" + "="*60)
    print("MISSING VALUES ANALYSIS")
    print("="*60)
    
    missing_count = df.isnull().sum()
    missing_percent = (missing_count / len(df)) * 100
    
    missing_df = pd.DataFrame({
        'Column': missing_count.index,
        'Missing_Count': missing_count.values,
        'Missing_Percent': missing_percent.values,
        'Data_Type': df.dtypes.values
    })
    missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Percent', ascending=False)
    
    if len(missing_df) > 0:
        print(f"Found {len(missing_df)} columns with missing values")
        display(missing_df)
        
        # Visualize missing values
        fig, ax = plt.subplots(figsize=(12, 6))
        missing_df.set_index('Column')['Missing_Percent'].plot(kind='bar', ax=ax)
        ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
        ax.set_xlabel('Column')
        ax.set_ylabel('Missing Percent (%)')
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print("✅ No missing values found in dataset")
    
    quality_report['missing_values'] = missing_df.to_dict() if len(missing_df) > 0 else {}
    
    # 2. Data types analysis
    print("\n" + "="*60)
    print("DATA TYPES ANALYSIS")
    print("="*60)
    
    dtype_df = pd.DataFrame({
        'Column': df.columns,
        'Data_Type': df.dtypes.values,
        'Unique_Values': [df[col].nunique() for col in df.columns],
        'Memory_MB': [df[col].memory_usage(deep=True) / 1024**2 for col in df.columns]
    })
    display(dtype_df)
    
    # Memory usage summary
    total_memory = dtype_df['Memory_MB'].sum()
    print(f"\nTotal memory usage: {total_memory:.2f} MB")
    
    quality_report['data_types'] = dtype_df.to_dict()
    
    # 3. Duplicate rows analysis
    print("\n" + "="*60)
    print("DUPLICATE ROWS ANALYSIS")
    print("="*60)
    
    duplicate_count = df.duplicated().sum()
    duplicate_percent = (duplicate_count / len(df)) * 100
    
    print(f"Duplicate rows: {duplicate_count:,} ({duplicate_percent:.4f}%)")
    
    if duplicate_count > 0:
        # Show sample of duplicates
        duplicate_mask = df.duplicated(keep='first')
        duplicate_samples = df[duplicate_mask].head(5)
        print("\nSample duplicate rows (first occurrence kept):")
        display(duplicate_samples)
    
    quality_report['duplicates'] = {
        'count': int(duplicate_count),
        'percent': float(duplicate_percent)
    }
    
    # 4. Zero values analysis (important for fraud detection)
    print("\n" + "="*60)
    print("ZERO VALUES ANALYSIS")
    print("="*60)
    
    zero_count = (df == 0).sum()
    zero_percent = (zero_count / len(df)) * 100
    
    zero_df = pd.DataFrame({
        'Column': zero_count.index,
        'Zero_Count': zero_count.values,
        'Zero_Percent': zero_percent.values
    })
    zero_df = zero_df[zero_df['Zero_Count'] > 0].sort_values('Zero_Percent', ascending=False)
    
    if len(zero_df) > 0:
        print("Columns with zero values:")
        display(zero_df)
        
        # Visualize zeros
        fig, ax = plt.subplots(figsize=(12, 6))
        zero_df.set_index('Column')['Zero_Percent'].plot(kind='bar', ax=ax, color='orange')
        ax.set_title('Zero Values by Column', fontsize=14, fontweight='bold')
        ax.set_xlabel('Column')
        ax.set_ylabel('Zero Percent (%)')
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    
    quality_report['zero_values'] = zero_df.to_dict() if len(zero_df) > 0 else {}
    
    # 5. Statistical summary
    print("\n" + "="*60)
    print("STATISTICAL SUMMARY")
    print("="*60)
    
    # Separate numerical and categorical columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    
    if len(numerical_cols) > 0:
        print("\nNumerical Features Summary:")
        display(df[numerical_cols].describe().T)
    
    if len(categorical_cols) > 0:
        print("\nCategorical Features Summary:")
        for col in categorical_cols:
            print(f"\n{col}:")
            print(df[col].value_counts().head())
    
    quality_report['numerical_summary'] = df[numerical_cols].describe().to_dict() if len(numerical_cols) > 0 else {}
    
    return quality_report

# Perform data quality assessment
quality_report = assess_data_quality(df)

# %% [markdown]
# ### 7. Target Variable Analysis (Class Imbalance)

# %%
def analyze_target_variable(df, target_col='Class'):
    """
    Comprehensive analysis of target variable distribution.
    
    Args:
        df (pd.DataFrame): Input dataframe
        target_col (str): Name of target column
    """
    print("\n" + "="*60)
    print("TARGET VARIABLE ANALYSIS (CLASS IMBALANCE)")
    print("="*60)
    
    # Class distribution
    class_counts = df[target_col].value_counts()
    class_percentages = df[target_col].value_counts(normalize=True) * 100
    
    # Create distribution dataframe
    dist_df = pd.DataFrame({
        'Class': class_counts.index,
        'Count': class_counts.values,
        'Percentage': class_percentages.values
    })
    dist_df['Class'] = dist_df['Class'].map({0: 'Non-Fraud (0)', 1: 'Fraud (1)'})
    
    print("\nClass Distribution:")
    display(dist_df)
    
    # Calculate imbalance metrics
    total_samples = len(df)
    minority_class = min(class_counts.index, key=lambda x: class_counts[x])
    majority_class = max(class_counts.index, key=lambda x: class_counts[x])
    
    imbalance_ratio = class_counts[majority_class] / class_counts[minority_class]
    
    print(f"\nImbalance Metrics:")
    print(f"  - Total samples: {total_samples:,}")
    print(f"  - Majority class ({majority_class}): {class_counts[majority_class]:,} ({class_percentages[majority_class]:.4f}%)")
    print(f"  - Minority class ({minority_class}): {class_counts[minority_class]:,} ({class_percentages[minority_class]:.4f}%)")
    print(f"  - Imbalance ratio: {imbalance_ratio:.2f}:1")
    
    # Visualizations
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Bar plot
    colors = ['#2ecc71', '#e74c3c']
    bars = axes[0].bar(dist_df['Class'], dist_df['Count'], color=colors)
    axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Count')
    
    # Add count labels on bars
    for bar, count in zip(bars, dist_df['Count']):
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{count:,}', ha='center', va='bottom')
    
    # Pie chart
    wedges, texts, autotexts = axes[1].pie(
        dist_df['Count'], 
        labels=dist_df['Class'],
        autopct='%1.4f%%',
        colors=colors,
        explode=(0, 0.1),  # Explode fraud slice
        shadow=True,
        startangle=90
    )
    axes[1].set_title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')
    
    # Logarithmic scale bar plot (to better visualize imbalance)
    axes[2].bar(dist_df['Class'], dist_df['Count'], color=colors)
    axes[2].set_yscale('log')
    axes[2].set_title('Class Distribution (Log Scale)', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Class')
    axes[2].set_ylabel('Count (log scale)')
    axes[2].grid(True, alpha=0.3, axis='y')
    
    # Add count labels
    for bar, count in zip(bars, dist_df['Count']):
        height = bar.get_height()
        axes[2].text(bar.get_x() + bar.get_width()/2., height,
                    f'{count:,}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Impact analysis
    print("\n" + "="*60)
    print("IMBALANCE IMPACT ANALYSIS")
    print("="*60)
    print("""
    Class imbalance is a critical challenge in fraud detection:
    
    1. Model Bias: Models tend to predict majority class, leading to poor fraud detection
    2. Evaluation Metrics: Accuracy becomes misleading (99.8% accuracy might mean missing all fraud)
    3. Learning Difficulty: Few fraud examples make pattern learning challenging
    
    Recommended Approaches:
    - Use appropriate metrics: Precision, Recall, F1-Score, AUC-PR
    - Apply resampling techniques: SMOTE, ADASYN, Random Undersampling
    - Use cost-sensitive learning: Weighted loss functions
    - Consider anomaly detection approaches: Isolation Forest, One-Class SVM
    """)
    
    return {
        'class_counts': class_counts.to_dict(),
        'class_percentages': class_percentages.to_dict(),
        'imbalance_ratio': imbalance_ratio
    }

# Analyze target variable
target_analysis = analyze_target_variable(df, 'Class')

# %% [markdown]
# ### 8. Feature Analysis - Time and Amount

# %%
def analyze_time_and_amount(df):
    """
    Analyze Time and Amount features in detail.
    
    Args:
        df (pd.DataFrame): Input dataframe
    """
    print("\n" + "="*60)
    print("TIME AND AMOUNT ANALYSIS")
    print("="*60)
    
    # Separate fraud and non-fraud data
    fraud_df = df[df['Class'] == 1]
    non_fraud_df = df[df['Class'] == 0]
    
    # Time analysis
    print("\n" + "-"*40)
    print("TIME FEATURE ANALYSIS")
    print("-"*40)
    
    # Convert seconds to hours for better interpretation
    df['Hour'] = (df['Time'] / 3600) % 24  # Convert to hour of day
    fraud_df['Hour'] = (fraud_df['Time'] / 3600) % 24
    non_fraud_df['Hour'] = (non_fraud_df['Time'] / 3600) % 24
    
    # Time statistics
    print("\nTime Statistics (seconds):")
    time_stats = pd.DataFrame({
        'Statistic': ['Mean', 'Std', 'Min', '25%', '50%', '75%', 'Max'],
        'All Transactions': [
            df['Time'].mean(),
            df['Time'].std(),
            df['Time'].min(),
            df['Time'].quantile(0.25),
            df['Time'].median(),
            df['Time'].quantile(0.75),
            df['Time'].max()
        ],
        'Non-Fraud': [
            non_fraud_df['Time'].mean(),
            non_fraud_df['Time'].std(),
            non_fraud_df['Time'].min(),
            non_fraud_df['Time'].quantile(0.25),
            non_fraud_df['Time'].median(),
            non_fraud_df['Time'].quantile(0.75),
            non_fraud_df['Time'].max()
        ],
        'Fraud': [
            fraud_df['Time'].mean(),
            fraud_df['Time'].std(),
            fraud_df['Time'].min(),
            fraud_df['Time'].quantile(0.25),
            fraud_df['Time'].median(),
            fraud_df['Time'].quantile(0.75),
            fraud_df['Time'].max()
        ]
    })
    display(time_stats)
    
    # Amount analysis
    print("\n" + "-"*40)
    print("AMOUNT FEATURE ANALYSIS")
    print("-"*40)
    
    print("\nAmount Statistics ($):")
    amount_stats = pd.DataFrame({
        'Statistic': ['Mean', 'Std', 'Min', '25%', '50%', '75%', 'Max'],
        'All Transactions': [
            df['Amount'].mean(),
            df['Amount'].std(),
            df['Amount'].min(),
            df['Amount'].quantile(0.25),
            df['Amount'].median(),
            df['Amount'].quantile(0.75),
            df['Amount'].max()
        ],
        'Non-Fraud': [
            non_fraud_df['Amount'].mean(),
            non_fraud_df['Amount'].std(),
            non_fraud_df['Amount'].min(),
            non_fraud_df['Amount'].quantile(0.25),
            non_fraud_df['Amount'].median(),
            non_fraud_df['Amount'].quantile(0.75),
            non_fraud_df['Amount'].max()
        ],
        'Fraud': [
            fraud_df['Amount'].mean(),
            fraud_df['Amount'].std(),
            fraud_df['Amount'].min(),
            fraud_df['Amount'].quantile(0.25),
            fraud_df['Amount'].median(),
            fraud_df['Amount'].quantile(0.75),
            fraud_df['Amount'].max()
        ]
    })
    display(amount_stats)
    
    # Visualizations
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('Time and Amount Analysis', fontsize=16, fontweight='bold')
    
    # 1. Time distribution by class (histogram)
    axes[0, 0].hist(non_fraud_df['Hour'], bins=48, alpha=0.7, label='Non-Fraud', density=True, color='blue')
    axes[0, 0].hist(fraud_df['Hour'], bins=48, alpha=0.7, label='Fraud', density=True, color='red')
    axes[0, 0].set_xlabel('Hour of Day')
    axes[0, 0].set_ylabel('Density')
    axes[0, 0].set_title('Transaction Time Distribution by Class')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Time boxplot by class
    box_data = [non_fraud_df['Hour'], fraud_df['Hour']]
    bp = axes[0, 1].boxplot(box_data, labels=['Non-Fraud', 'Fraud'], patch_artist=True)
    bp['boxes'][0].set_facecolor('blue')
    bp['boxes'][1].set_facecolor('red')
    axes[0, 1].set_ylabel('Hour of Day')
    axes[0, 1].set_title('Time Distribution Boxplot by Class')
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # 3. Amount distribution (log scale)
    axes[0, 2].hist(non_fraud_df['Amount'], bins=50, alpha=0.7, label='Non-Fraud', density=True, color='blue')
    axes[0, 2].hist(fraud_df['Amount'], bins=50, alpha=0.7, label='Fraud', density=True, color='red')
    axes[0, 2].set_xscale('log')
    axes[0, 2].set_xlabel('Amount ($) - Log Scale')
    axes[0, 2].set_ylabel('Density')
    axes[0, 2].set_title('Transaction Amount Distribution by Class')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # 4. Amount boxplot by class
    box_data = [non_fraud_df['Amount'], fraud_df['Amount']]
    bp = axes[1, 0].boxplot(box_data, labels=['Non-Fraud', 'Fraud'], patch_artist=True)
    bp['boxes'][0].set_facecolor('blue')
    bp['boxes'][1].set_facecolor('red')
    axes[1, 0].set_ylabel('Amount ($)')
    axes[1, 0].set_title('Amount Distribution Boxplot by Class')
    axes[1, 0].set_yscale('log')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # 5. Amount vs Time scatter
    axes[1, 1].scatter(non_fraud_df['Hour'], non_fraud_df['Amount'], 
                       alpha=0.1, s=1, label='Non-Fraud', color='blue')
    axes[1, 1].scatter(fraud_df['Hour'], fraud_df['Amount'], 
                       alpha=0.5, s=10, label='Fraud', color='red')
    axes[1, 1].set_xlabel('Hour of Day')
    axes[1, 1].set_ylabel('Amount ($)')
    axes[1, 1].set_yscale('log')
    axes[1, 1].set_title('Amount vs Time by Class')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # 6. Fraud rate by hour
    fraud_rate_by_hour = df.groupby('Hour')['Class'].mean() * 100
    axes[1, 2].plot(fraud_rate_by_hour.index, fraud_rate_by_hour.values, 
                    marker='o', color='red', linewidth=2)
    axes[1, 2].set_xlabel('Hour of Day')
    axes[1, 2].set_ylabel('Fraud Rate (%)')
    axes[1, 2].set_title('Fraud Rate by Hour of Day')
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].axhline(y=fraud_rate_by_hour.mean(), color='blue', linestyle='--', 
                       label=f'Average: {fraud_rate_by_hour.mean():.4f}%')
    axes[1, 2].legend()
    
    plt.tight_layout()
    plt.show()
    
    return {
        'time_stats': time_stats.to_dict(),
        'amount_stats': amount_stats.to_dict(),
        'fraud_rate_by_hour': fraud_rate_by_hour.to_dict()
    }

# Analyze time and amount
time_amount_analysis = analyze_time_and_amount(df)

# %% [markdown]
# ### 9. Feature Analysis - PCA Components (V1-V28)

# %%
def analyze_pca_features(df):
    """
    Analyze PCA-transformed features (V1-V28).
    
    Args:
        df (pd.DataFrame): Input dataframe
    """
    print("\n" + "="*60)
    print("PCA FEATURES ANALYSIS (V1-V28)")
    print("="*60)
    
    # Get PCA feature columns
    pca_cols = [col for col in df.columns if col.startswith('V')]
    
    # Separate fraud and non-fraud
    fraud_df = df[df['Class'] == 1]
    non_fraud_df = df[df['Class'] == 0]
    
    # Calculate statistics for each PCA feature
    pca_stats = []
    for col in pca_cols:
        stats = {
            'Feature': col,
            'Non-Fraud Mean': non_fraud_df[col].mean(),
            'Fraud Mean': fraud_df[col].mean(),
            'Mean Difference': abs(fraud_df[col].mean() - non_fraud_df[col].mean()),
            'Non-Fraud Std': non_fraud_df[col].std(),
            'Fraud Std': fraud_df[col].std(),
            'Std Ratio': fraud_df[col].std() / non_fraud_df[col].std() if non_fraud_df[col].std() > 0 else 0
        }
        pca_stats.append(stats)
    
    pca_stats_df = pd.DataFrame(pca_stats)
    pca_stats_df = pca_stats_df.sort_values('Mean Difference', ascending=False)
    
    print("\nPCA Features Statistics (sorted by mean difference):")
    display(pca_stats_df)
    
    # Visualizations
    fig, axes = plt.subplots(3, 3, figsize=(20, 18))
    fig.suptitle('PCA Features Analysis - Top 9 Most Discriminative Features', fontsize=16, fontweight='bold')
    
    # Get top 9 features by mean difference
    top_features = pca_stats_df.head(9)['Feature'].tolist()
    
    for idx, feature in enumerate(top_features):
        row = idx // 3
        col = idx % 3
        
        # Plot distribution by class
        axes[row, col].hist(non_fraud_df[feature], bins=50, alpha=0.7, 
                           label='Non-Fraud', density=True, color='blue')
        axes[row, col].hist(fraud_df[feature], bins=50, alpha=0.7, 
                           label='Fraud', density=True, color='red')
        axes[row, col].set_xlabel(feature)
        axes[row, col].set_ylabel('Density')
        axes[row, col].set_title(f'{feature} Distribution by Class')
        axes[row, col].legend()
        axes[row, col].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Correlation analysis
    print("\n" + "-"*40)
    print("CORRELATION ANALYSIS")
    print("-"*40)
    
    # Calculate correlation with target
    correlations = df[pca_cols + ['Class']].corr()['Class'].drop('Class').sort_values(ascending=False)
    
    print("\nTop 10 Features Correlated with Fraud:")
    top_corr = correlations.head(10)
    for feature, corr in top_corr.items():
        print(f"  {feature}: {corr:.4f}")
    
    print("\nBottom 10 Features Correlated with Fraud:")
    bottom_corr = correlations.tail(10)
    for feature, corr in bottom_corr.items():
        print(f"  {feature}: {corr:.4f}")
    
    # Visualize correlation matrix
    fig, ax = plt.subplots(figsize=(14, 12))
    
    # Select top features for correlation matrix
    top_n = 15
    top_features_corr = correlations.head(top_n).index.tolist() + correlations.tail(top_n).index.tolist()
    corr_matrix = df[top_features_corr + ['Class']].corr()
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, linewidths=1, ax=ax)
    ax.set_title(f'Correlation Matrix - Top {top_n} Features with Class', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'pca_stats': pca_stats_df.to_dict(),
        'correlations': correlations.to_dict()
    }

# Analyze PCA features
pca_analysis = analyze_pca_features(df)

# %% [markdown]
# ### 10. Statistical Hypothesis Testing

# %%
def hypothesis_testing(df):
    """
    Perform statistical hypothesis tests to validate feature significance.
    
    Args:
        df (pd.DataFrame): Input dataframe
    """
    from scipy import stats
    
    print("\n" + "="*60)
    print("STATISTICAL HYPOTHESIS TESTING")
    print("="*60)
    
    # Separate fraud and non-fraud
    fraud_df = df[df['Class'] == 1]
    non_fraud_df = df[df['Class'] == 0]
    
    results = []
    
    # Test for each numerical feature
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    numerical_cols = [col for col in numerical_cols if col != 'Class']
    
    for col in numerical_cols:
        # Skip if not enough samples
        if len(fraud_df[col]) < 2 or len(non_fraud_df[col]) < 2:
            continue
        
        # Check normality using Shapiro-Wilk test (for both groups)
        _, p_normal_fraud = stats.shapiro(fraud_df[col].sample(min(5000, len(fraud_df[col]))))
        _, p_normal_nonfraud = stats.shapiro(non_fraud_df[col].sample(min(5000, len(non_fraud_df[col]))))
        
        # If data is approximately normal, use t-test; otherwise use Mann-Whitney U
        if p_normal_fraud > 0.05 and p_normal_nonfraud > 0.05:
            # Parametric test: t-test
            t_stat, p_value = stats.ttest_ind(fraud_df[col], non_fraud_df[col], equal_var=False)
            test_used = "Welch's t-test"
        else:
            # Non-parametric test: Mann-Whitney U
            u_stat, p_value = stats.mannwhitneyu(fraud_df[col], non_fraud_df[col], alternative='two-sided')
            t_stat = u_stat
            test_used = "Mann-Whitney U"
        
        # Calculate effect size (Cohen's d)
        mean1, mean2 = fraud_df[col].mean(), non_fraud_df[col].mean()
        std1, std2 = fraud_df[col].std(), non_fraud_df[col].std()
        
        # Pooled standard deviation
        n1, n2 = len(fraud_df[col]), len(non_fraud_df[col])
        pooled_std = np.sqrt(((n1 - 1) * std1**2 + (n2 - 1) * std2**2) / (n1 + n2 - 2))
        cohens_d = abs(mean1 - mean2) / pooled_std if pooled_std > 0 else 0
        
        results.append({
            'Feature': col,
            'Test Used': test_used,
            'Statistic': t_stat,
            'P-Value': p_value,
            'Significant': p_value < 0.05,
            'Effect Size (Cohen\'s d)': cohens_d,
            'Effect Magnitude': interpret_effect_size(cohens_d)
        })
    
    # Create results dataframe
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('P-Value')
    
    print("\nHypothesis Test Results (sorted by p-value):")
    display(results_df)
    
    # Summary statistics
    print("\n" + "-"*40)
    print("SUMMARY STATISTICS")
    print("-"*40)
    print(f"Total features tested: {len(results_df)}")
    print(f"Features with significant differences (p < 0.05): {results_df['Significant'].sum()}")
    print(f"Features with p < 0.001: {(results_df['P-Value'] < 0.001).sum()}")
    
    print("\nEffect Size Distribution:")
    effect_dist = results_df['Effect Magnitude'].value_counts()
    for effect, count in effect_dist.items():
        print(f"  {effect}: {count}")
    
    # Visualize results
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. P-value distribution (log scale)
    axes[0].hist(results_df['P-Value'], bins=30, edgecolor='black', alpha=0.7)
    axes[0].axvline(x=0.05, color='red', linestyle='--', label='α = 0.05')
    axes[0].set_xlabel('P-Value')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of P-Values')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. -log10(p-value) (higher is more significant)
    results_df['-log10(p)'] = -np.log10(results_df['P-Value'] + 1e-10)  # Add small constant to avoid log(0)
    top_features = results_df.nlargest(20, '-log10(p)')
    
    axes[1].barh(top_features['Feature'], top_features['-log10(p)'], color='skyblue')
    axes[1].axvline(x=-np.log10(0.05), color='red', linestyle='--', label='p = 0.05')
    axes[1].set_xlabel('-log10(P-Value)')
    axes[1].set_title('Top 20 Features by Statistical Significance')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='x')
    
    # 3. Effect size distribution
    axes[2].hist(results_df['Effect Size (Cohen\'s d)'], bins=30, edgecolor='black', alpha=0.7)
    axes[2].axvline(x=0.2, color='green', linestyle='--', label='Small')
    axes[2].axvline(x=0.5, color='orange', linestyle='--', label='Medium')
    axes[2].axvline(x=0.8, color='red', linestyle='--', label='Large')
    axes[2].set_xlabel("Cohen's d")
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('Distribution of Effect Sizes')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return results_df

def interpret_effect_size(d):
    """
    Interpret Cohen's d effect size.
    
    Args:
        d (float): Cohen's d value
        
    Returns:
        str: Interpretation
    """
    if d < 0.2:
        return "Negligible"
    elif d < 0.5:
        return "Small"
    elif d < 0.8:
        return "Medium"
    else:
        return "Large"

# Perform hypothesis testing
hypothesis_results = hypothesis_testing(df)

# %% [markdown]
# ### 11. Outlier Detection

# %%
def detect_outliers(df):
    """
    Detect outliers using multiple methods.
    
    Args:
        df (pd.DataFrame): Input dataframe
    """
    from sklearn.ensemble import IsolationForest
    from sklearn.covariance import EllipticEnvelope
    
    print("\n" + "="*60)
    print("OUTLIER DETECTION ANALYSIS")
    print("="*60)
    
    # Select numerical columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    numerical_cols = [col for col in numerical_cols if col != 'Class']
    
    outlier_results = {}
    
    # Method 1: IQR Method
    print("\n" + "-"*40)
    print("METHOD 1: IQR (Interquartile Range)")
    print("-"*40)
    
    iqr_outliers = {}
    for col in numerical_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_count = len(outliers)
        outlier_percent = (outlier_count / len(df)) * 100
        
        iqr_outliers[col] = {
            'count': outlier_count,
            'percent': outlier_percent,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound
        }
    
    iqr_df = pd.DataFrame(iqr_outliers).T
    iqr_df = iqr_df.sort_values('percent', ascending=False)
    print("\nOutliers detected by IQR method:")
    display(iqr_df.head(10))
    
    outlier_results['iqr'] = iqr_outliers
    
    # Method 2: Z-Score Method
    print("\n" + "-"*40)
    print("METHOD 2: Z-Score (3 sigma)")
    print("-"*40)
    
    from scipy import stats
    
    zscore_outliers = {}
    for col in numerical_cols:
        z_scores = np.abs(stats.zscore(df[col].dropna()))
        outlier_count = (z_scores > 3).sum()
        outlier_percent = (outlier_count / len(df)) * 100
        
        zscore_outliers[col] = {
            'count': outlier_count,
            'percent': outlier_percent,
            'threshold': 3
        }
    
    zscore_df = pd.DataFrame(zscore_outliers).T
    zscore_df = zscore_df.sort_values('percent', ascending=False)
    print("\nOutliers detected by Z-Score method (|z| > 3):")
    display(zscore_df.head(10))
    
    outlier_results['zscore'] = zscore_outliers
    
    # Method 3: Modified Z-Score (robust to non-normal distributions)
    print("\n" + "-"*40)
    print("METHOD 3: Modified Z-Score (using MAD)")
    print("-"*40)
    
    modified_zscore_outliers = {}
    for col in numerical_cols:
        median = df[col].median()
        mad = np.median(np.abs(df[col] - median))
        
        if mad > 0:
            modified_z_scores = 0.6745 * (df[col] - median) / mad
            outlier_count = (np.abs(modified_z_scores) > 3.5).sum()
            outlier_percent = (outlier_count / len(df)) * 100
        else:
            outlier_count = 0
            outlier_percent = 0
        
        modified_zscore_outliers[col] = {
            'count': outlier_count,
            'percent': outlier_percent,
            'threshold': 3.5
        }
    
    modified_df = pd.DataFrame(modified_zscore_outliers).T
    modified_df = modified_df.sort_values('percent', ascending=False)
    print("\nOutliers detected by Modified Z-Score method (|M| > 3.5):")
    display(modified_df.head(10))
    
    outlier_results['modified_zscore'] = modified_zscore_outliers
    
    # Method 4: Isolation Forest (multivariate)
    print("\n" + "-"*40)
    print("METHOD 4: Isolation Forest (Multivariate)")
    print("-"*40)
    
    # Use a sample for Isolation Forest (due to computational cost)
    sample_size = min(50000, len(df))
    sample_df = df.sample(sample_size, random_state=42)
    
    iso_forest = IsolationForest(contamination=0.1, random_state=42, n_estimators=100)
    outlier_labels = iso_forest.fit_predict(sample_df[numerical_cols])
    
    # -1 for outliers, 1 for inliers
    outlier_count = (outlier_labels == -1).sum()
    outlier_percent = (outlier_count / sample_size) * 100
    
    print(f"\nIsolation Forest detected {outlier_count} outliers ({outlier_percent:.2f}% of sample)")
    
    outlier_results['isolation_forest'] = {
        'count': outlier_count,
        'percent': outlier_percent,
        'sample_size': sample_size
    }
    
    # Visualize outliers for top features
    print("\n" + "-"*40)
    print("OUTLIER VISUALIZATION")
    print("-"*40)
    
    # Get top features with most outliers
    top_outlier_features = iqr_df.head(4).index.tolist()
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, feature in enumerate(top_outlier_features):
        # Create boxplot with outliers highlighted
        ax = axes[idx]
        
        # Plot distribution
        ax.hist(df[feature], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
        
        # Highlight outliers based on IQR
        Q1 = df[feature].quantile(0.25)
        Q3 = df[feature].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)]
        
        # Plot outliers in red
        ax.scatter(outliers[feature], [0] * len(outliers), color='red', s=10, alpha=0.5, label='Outliers')
        
        ax.set_xlabel(feature)
        ax.set_ylabel('Frequency')
        ax.set_title(f'{feature} - Outliers Highlighted (IQR method)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Add vertical lines for bounds
        ax.axvline(x=lower_bound, color='green', linestyle='--', alpha=0.7, label='Lower bound')
        ax.axvline(x=upper_bound, color='green', linestyle='--', alpha=0.7, label='Upper bound')
    
    plt.tight_layout()
    plt.show()
    
    # Impact analysis
    print("\n" + "-"*40)
    print("IMPACT OF OUTLIERS ON FRAUD DETECTION")
    print("-"*40)
    print("""
    Outliers in fraud detection can be either:
    
    1. Genuine Fraud: Large transactions, unusual patterns - THESE ARE IMPORTANT TO KEEP
    2. Data Errors: Recording mistakes, system errors - THESE SHOULD BE REMOVED
    3. Extreme but Legitimate: Wealthy customers, business transactions - THESE NEED SPECIAL HANDLING
    
    Recommendations:
    - Don't automatically remove outliers as they may contain fraud cases
    - Use robust scaling methods (RobustScaler) that are less sensitive to outliers
    - Consider winsorization to cap extreme values
    - Investigate outliers with high fraud probability
    """)
    
    return outlier_results

# Detect outliers
outlier_results = detect_outliers(df)

# %% [markdown]
# ### 12. Data Quality Report Generation

# %%
def generate_quality_report(df, quality_report, target_analysis, time_amount_analysis, 
                           pca_analysis, hypothesis_results, outlier_results):
    """
    Generate comprehensive data quality report.
    
    Args:
        df (pd.DataFrame): Input dataframe
        quality_report (dict): Data quality metrics
        target_analysis (dict): Target variable analysis
        time_amount_analysis (dict): Time and amount analysis
        pca_analysis (dict): PCA features analysis
        hypothesis_results (pd.DataFrame): Hypothesis test results
        outlier_results (dict): Outlier detection results
    """
    print("\n" + "="*60)
    print("DATA QUALITY REPORT")
    print("="*60)
    print(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Dataset: {config.get('data_sources', {}).get('transactions', {}).get('path', 'Unknown')}")
    
    # 1. Dataset Overview
    print("\n" + "-"*40)
    print("1. DATASET OVERVIEW")
    print("-"*40)
    print(f"Total Transactions: {len(df):,}")
    print(f"Number of Features: {df.shape[1]}")
    print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"Fraud Cases: {target_analysis['class_counts'][1]:,} ({target_analysis['class_percentages'][1]:.4f}%)")
    print(f"Non-Fraud Cases: {target_analysis['class_counts'][0]:,} ({target_analysis['class_percentages'][0]:.4f}%)")
    print(f"Imbalance Ratio: {target_analysis['imbalance_ratio']:.2f}:1")
    
    # 2. Data Quality Metrics
    print("\n" + "-"*40)
    print("2. DATA QUALITY METRICS")
    print("-"*40)
    
    missing_cols = len(quality_report.get('missing_values', {}).get('Column', []))
    print(f"Columns with Missing Values: {missing_cols}")
    
    duplicate_count = quality_report.get('duplicates', {}).get('count', 0)
    duplicate_percent = quality_report.get('duplicates', {}).get('percent', 0)
    print(f"Duplicate Rows: {duplicate_count:,} ({duplicate_percent:.4f}%)")
    
    # 3. Feature Statistics
    print("\n" + "-"*40)
    print("3. FEATURE STATISTICS")
    print("-"*40)
    
    # Top discriminative features
    if pca_analysis:
        pca_stats = pca_analysis.get('pca_stats', {})
        if pca_stats:
            print("\nTop 5 Most Discriminative Features (by mean difference):")
            top_features = sorted(pca_stats.items(), key=lambda x: x[1].get('Mean Difference', 0), reverse=True)[:5]
            for feature, stats in top_features:
                print(f"  {feature}: diff={stats.get('Mean Difference', 0):.4f}")
    
    # 4. Statistical Significance
    print("\n" + "-"*40)
    print("4. STATISTICAL SIGNIFICANCE")
    print("-"*40)
    
    if hypothesis_results is not None:
        sig_features = hypothesis_results[hypothesis_results['Significant'] == True]
        print(f"Features with significant differences (p < 0.05): {len(sig_features)}/{len(hypothesis_results)}")
        
        print("\nTop 5 Most Significant Features:")
        top_sig = hypothesis_results.nsmallest(5, 'P-Value')[['Feature', 'P-Value', 'Effect Size (Cohen\'s d)']]
        for _, row in top_sig.iterrows():
            print(f"  {row['Feature']}: p={row['P-Value']:.4e}, d={row['Effect Size (Cohen\'s d)']:.4f}")
    
    # 5. Outlier Summary
    print("\n" + "-"*40)
    print("5. OUTLIER SUMMARY")
    print("-"*40)
    
    if outlier_results:
        iqr_outliers = outlier_results.get('iqr', {})
        if iqr_outliers:
            # Calculate average outlier percentage
            outlier_percents = [stats.get('percent', 0) for stats in iqr_outliers.values()]
            avg_outlier_percent = np.mean(outlier_percents)
            max_outlier_feature = max(iqr_outliers.items(), key=lambda x: x[1].get('percent', 0))
            
            print(f"Average outlier percentage (IQR method): {avg_outlier_percent:.2f}%")
            print(f"Feature with most outliers: {max_outlier_feature[0]} ({max_outlier_feature[1].get('percent', 0):.2f}%)")
    
    # 6. Recommendations
    print("\n" + "-"*40)
    print("6. RECOMMENDATIONS")
    print("-"*40)
    
    recommendations = []
    
    # Class imbalance recommendation
    if target_analysis['imbalance_ratio'] > 10:
        recommendations.append("🔴 SEVERE CLASS IMBALANCE: Use SMOTE, weighted loss functions, or anomaly detection")
    
    # Missing values recommendation
    if missing_cols > 0:
        recommendations.append("🟡 MISSING VALUES: Implement appropriate imputation strategy")
    
    # Outliers recommendation
    if avg_outlier_percent > 5:
        recommendations.append("🟡 SIGNIFICANT OUTLIERS: Use robust scaling methods (RobustScaler)")
    
    # Duplicates recommendation
    if duplicate_count > 0:
        recommendations.append("🟡 DUPLICATE RECORDS: Remove duplicates or investigate data collection issues")
    
    if not recommendations:
        recommendations.append("✅ DATA QUALITY IS GOOD: Proceed to feature engineering")
    
    for rec in recommendations:
        print(rec)
    
    # Save report
    report = {
        'generated_at': datetime.now().isoformat(),
        'dataset_overview': {
            'total_transactions': len(df),
            'total_features': df.shape[1],
            'memory_mb': df.memory_usage(deep=True).sum() / 1024**2,
            'fraud_count': target_analysis['class_counts'][1],
            'fraud_percent': target_analysis['class_percentages'][1],
            'imbalance_ratio': target_analysis['imbalance_ratio']
        },
        'quality_metrics': quality_report,
        'recommendations': recommendations
    }
    
    # Save to file
    report_path = '../artifacts/reports/data_quality_report.json'
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    
    print(f"\n✅ Report saved to {report_path}")

# Generate comprehensive report
generate_quality_report(df, quality_report, target_analysis, time_amount_analysis,
                       pca_analysis, hypothesis_results, outlier_results)

# %% [markdown]
# ### 13. Save Processed Data

# %%
def save_processed_data(df):
    """
    Save processed data in multiple formats for future use.
    
    Args:
        df (pd.DataFrame): Processed dataframe
    """
    print("\n" + "="*60)
    print("SAVING PROCESSED DATA")
    print("="*60)
    
    # Create directories if they don't exist
    os.makedirs('../data/processed', exist_ok=True)
    os.makedirs('../data/cache', exist_ok=True)
    
    # Save as Parquet (efficient format)
    parquet_path = '../data/processed/transactions_processed.parquet'
    df.to_parquet(parquet_path, index=False, engine='pyarrow')
    print(f"✅ Saved to Parquet: {parquet_path}")
    
    # Save as CSV (interoperable format)
    csv_path = '../data/processed/transactions_processed.csv'
    df.to_csv(csv_path, index=False)
    print(f"✅ Saved to CSV: {csv_path}")
    
    # Save feature names for reference
    feature_info = {
        'numerical_features': df.select_dtypes(include=[np.number]).columns.tolist(),
        'categorical_features': df.select_dtypes(include=['object', 'category']).columns.tolist(),
        'total_features': df.shape[1],
        'total_samples': len(df)
    }
    
    info_path = '../data/processed/feature_info.json'
    with open(info_path, 'w') as f:
        json.dump(feature_info, f, indent=2)
    print(f"✅ Saved feature info: {info_path}")
    
    # Save sample for quick loading
    sample_path = '../data/cache/transactions_sample.parquet'
    df.sample(min(10000, len(df)), random_state=42).to_parquet(sample_path, index=False)
    print(f"✅ Saved sample: {sample_path}")
    
    return feature_info

# Save processed data
feature_info = save_processed_data(df)

# %% [markdown]
# ### 14. Summary and Next Steps

# %%
print("\n" + "="*60)
print("NOTEBOOK 01 SUMMARY")
print("="*60)
print("""
✅ COMPLETED TASKS:
1. Data Loading: Successfully loaded transaction data
2. Schema Validation: Verified data structure meets requirements
3. Data Quality Assessment: Analyzed missing values, duplicates, data types
4. Class Imbalance Analysis: Quantified and visualized fraud distribution
5. Feature Analysis: Detailed analysis of Time, Amount, and PCA features
6. Statistical Testing: Performed hypothesis tests for feature significance
7. Outlier Detection: Identified outliers using multiple methods
8. Quality Report: Generated comprehensive data quality report
9. Data Persistence: Saved processed data in multiple formats

📊 KEY FINDINGS:
""")

# Print key findings
print(f"- Dataset contains {len(df):,} transactions with {df.shape[1]} features")
print(f"- Fraud rate: {target_analysis['class_percentages'][1]:.4f}%")
print(f"- Imbalance ratio: {target_analysis['imbalance_ratio']:.2f}:1")

if hypothesis_results is not None:
    sig_count = hypothesis_results['Significant'].sum()
    print(f"- {sig_count} features show statistically significant differences between fraud/non-fraud")

print("\n" + "-"*40)
print("NEXT STEPS")
print("-"*40)
print("""
Proceed to Notebook 02: EDA & Statistical Analysis
- Detailed visualization of feature distributions
- Correlation analysis and feature selection
- Temporal pattern analysis
- Geographic analysis (if location data available)
- Behavioral pattern extraction
""")

print("\n" + "="*60)
print("END OF NOTEBOOK 01")
print("="*60)